# Combine 2025's ad spend

*Data Analysis in Python · Topic 5.1*

The media team wants one table of 2025's ad spend across Google Ads, Meta and YouTube, so every report this year starts from the same numbers.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that reads the three exports into the `google`, `meta` and `youtube` tables.
2. In the first cell marked ✏️, create three tables, `google_std`, `meta_std` and `youtube_std`, each with the standard columns in `COLUMNS` that its platform has, in that order:
   - `platform`: `"Google Ads"`, `"Meta"` or `"YouTube"` on every row
   - `spend`: the cost as a number of dollars (Google Ads' `cost_micros` counts millionths of a dollar)
   - `ad_set`: Meta's ad set name. Google Ads and YouTube have no ad sets, so their tables have no `ad_set` column.
3. In the second cell marked ✏️, create the `spend` table: the three tables stacked, numbered from 0, with the columns in `COLUMNS`, and only the rows dated in 2025.
4. In the last cell marked ✏️, create:
   - `spend_by_platform`: a Series of each platform's total spend, with the platforms as the index
   - `total_spend`: the three platforms' spend together, rounded to 2 decimals
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `spend` table has 10,754 rows and the columns `date`, `platform`, `campaign`, `ad_set`, `impressions`, `clicks` and `spend`, and its `ad_set` column is missing on the 4,550 Google Ads and YouTube rows. The `spend_by_platform` Series is Google Ads $198,630.66, Meta $78,756.62 and YouTube $41,317.80, and `total_spend` is `318705.08`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
COLUMNS = ["date", "platform", "campaign", "ad_set", "impressions", "clicks", "spend"]

In [ ]:
google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts", "Reporting ends"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
google.shape, meta.shape, youtube.shape

## One shape per platform

In [ ]:
# ✏️ Create google_std, meta_std and youtube_std


## One spend table

In [ ]:
# ✏️ Create spend: the three tables stacked, 2025 only


In [ ]:
# ✏️ Create spend_by_platform and total_spend


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    standard = ["date", "platform", "campaign", "ad_set", "impressions", "clicks", "spend"]
    tables = {"google_std": ("Google Ads", 7602), "meta_std": ("Meta", 12424), "youtube_std": ("YouTube", 1510)}
    problems = []
    for name, (platform, rows) in tables.items():
        table = found.get(name)
        expected_columns = [column for column in standard if column != "ad_set" or platform == "Meta"]
        if not isinstance(table, pd.DataFrame):
            problems.append(f"{name} isn't a DataFrame yet. Rename the export's columns with rename(columns={{...}}), add platform and spend, and keep the standard columns.")
        elif len(table) != rows:
            problems.append(f"The {name} table has {len(table):,} rows, but the {platform} export has {rows:,}. Keep every row here: the 2025 filter comes in the next cell.")
        elif extra := [column for column in table.columns if column not in standard]:
            problems.append(f"The {name} table has columns that aren't standard: {extra}. Rename the ones that are standard facts, and leave the rest out.")
        elif platform != "Meta" and "ad_set" in table.columns:
            problems.append(f"The {name} table has an ad_set column, but {platform} has no ad sets. Leave it out: pd.concat fills it with NaN when you stack the tables.")
        elif missing := [column for column in expected_columns if column not in table.columns]:
            problems.append(f"The {name} table has no {', '.join(missing)} column. {platform}'s export has it under another name: rename it.")
        elif list(table.columns) != expected_columns:
            problems.append(f"The {name} table's columns should be in COLUMNS' order: {expected_columns}. They are {list(table.columns)}.")
        elif set(table["platform"].unique()) != {platform}:
            problems.append(f"The {name} table's platform column should be {platform!r} on every row. It has {sorted(table['platform'].unique())}.")
        elif not pd.api.types.is_datetime64_any_dtype(table["date"]):
            problems.append(f"The {name} table's date column should hold dates. Use the export's date column, read as dates.")
    google_std = found.get("google_std")
    if problems:
        results.append((False, problems[0]))
    elif abs(google_std["spend"].sum() - 357599940000) < 1:
        results.append((False, "The google_std table's spend is still in micros: 357,599,940,000. Divide cost_micros by 1_000_000."))
    else:
        results.append((True, "google_std, meta_std and youtube_std have the standard columns, under one set of names."))

    spend = found.get("spend")
    if not isinstance(spend, pd.DataFrame):
        results.append((False, "spend isn't a DataFrame yet. Stack the three tables with pd.concat([...], ignore_index=True), keep the columns in COLUMNS, and keep the rows dated in 2025."))
    elif list(spend.columns) != standard:
        results.append((False, f"The spend table's columns should be COLUMNS, in that order. They are {list(spend.columns)}. Select them after stacking: pd.concat([...])[COLUMNS]."))
    elif len(spend) == 21536:
        results.append((False, "The spend table has 21,536 rows: both years. Keep only the rows dated in 2025."))
    elif len(spend) != 10754:
        results.append((False, f"The spend table has {len(spend):,} rows, but 2025 has 10,754: 3,796 from Google Ads, 6,204 from Meta and 754 from YouTube."))
    elif list(spend.index) != list(range(len(spend))):
        results.append((False, "The spend table should be numbered from 0. Use ignore_index=True when you stack, and reset_index(drop=True) after you filter."))
    elif empty := [column for column in standard if column != "ad_set" and spend[column].isna().any()]:
        results.append((False, f"The spend table has missing values in {', '.join(empty)}: a platform's column wasn't renamed, so pd.concat filled it with NaN. Check each platform's rename() dictionary."))
    elif spend["ad_set"].isna().sum() != 4550:
        results.append((False, f"The spend table's ad_set column is missing on {spend['ad_set'].isna().sum():,} rows, but it should be missing on the 4,550 Google Ads and YouTube rows only."))
    elif abs(spend["spend"].sum() - 318705.08) > 0.01:
        results.append((False, f"The spend table's spend adds up to {spend['spend'].sum():,.2f}, but 2025's three platforms spent 318,705.08. Check each platform's total against its export."))
    else:
        results.append((True, "The spend table has 2025's 10,754 rows from the three platforms, in the standard columns."))

    by_platform = found.get("spend_by_platform")
    expected = {"Google Ads": 198630.66, "Meta": 78756.62, "YouTube": 41317.80}
    if isinstance(by_platform, pd.DataFrame):
        results.append((False, "spend_by_platform is a DataFrame. Pick the spend column before summing, so it's a Series: spend.groupby(\"platform\")[\"spend\"].sum()."))
    elif not isinstance(by_platform, pd.Series):
        results.append((False, "spend_by_platform isn't a Series yet. Group the spend table by platform and total its spend column."))
    elif sorted(by_platform.index) != sorted(expected):
        results.append((False, f"The spend_by_platform Series should have the platforms Google Ads, Meta and YouTube as its index. It has {list(by_platform.index)}."))
    elif any(abs(by_platform[platform] - value) > 0.01 for platform, value in expected.items()):
        results.append((False, "The spend_by_platform Series has the three platforms, but not 2025's totals: Google Ads $198,630.66, Meta $78,756.62 and YouTube $41,317.80."))
    else:
        results.append((True, "2025's spend: Google Ads $198,630.66, Meta $78,756.62 and YouTube $41,317.80, each the same as its own export."))

    total = found.get("total_spend")
    if isinstance(total, (pd.DataFrame, pd.Series)):
        results.append((False, f"total_spend is a {type(total).__name__}, not a number. Total one column: round(spend[\"spend\"].sum(), 2)."))
    elif isinstance(total, str):
        results.append((False, f"total_spend is the text {total!r}. Keep it a number, 318705.08, and format it only when you show it."))
    elif pd.api.types.is_number(total) and abs(total - 318705.08) < 0.005:
        results.append((True, "The three platforms spent $318,705.08 in 2025."))
    else:
        results.append((False, f"total_spend should be round(spend[\"spend\"].sum(), 2), 318705.08. It is {total!r}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def standard_columns(table):
    """Return the table with only the standard columns it has, in the standard order."""
    return table[[column for column in COLUMNS if column in table.columns]]


google_std = google.rename(columns={"campaign_name": "campaign"})
google_std["platform"] = "Google Ads"
google_std["spend"] = google_std["cost_micros"] / 1_000_000
google_std = standard_columns(google_std)

meta_std = meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Ad set name": "ad_set",
                                "Impressions": "impressions", "Link clicks": "clicks", "Amount spent (USD)": "spend"})
meta_std["platform"] = "Meta"
meta_std = standard_columns(meta_std)

youtube_std = youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Impr.": "impressions", "Clicks": "clicks", "Cost": "spend"})
youtube_std["platform"] = "YouTube"
youtube_std = standard_columns(youtube_std)
list(google_std.columns), list(meta_std.columns), list(youtube_std.columns)
```

```python
spend = pd.concat([google_std, meta_std, youtube_std], ignore_index=True)[COLUMNS]
spend = spend[spend["date"].dt.strftime("%Y") == "2025"].reset_index(drop=True)
spend.shape, spend["ad_set"].isna().sum()
```

```python
spend_by_platform = spend.groupby("platform")["spend"].sum()
total_spend = round(spend["spend"].sum(), 2)
spend_by_platform, total_spend
```

</details>